# Predictive Maintenance XGBoost Classifier

Modern manufacturing facilities use sensor telemetry to predict equipment failures before they happen. This project constructs a synthetic industrial telemetry dataset containing speed, torque, tool wear, and temperature sensor logs, trains an XGBoost classification pipeline to detect failure risks, and evaluates model predictions.



In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc

# Generate synthetic sensor telemetry records (2,000 machines/samples)
np.random.seed(54)
n_samples = 2000

# Base continuous features
rpm = np.random.normal(1500, 200, n_samples).clip(800, 2200)
torque = np.random.normal(40, 10, n_samples).clip(10, 80)
tool_wear = np.random.uniform(0, 240, n_samples)  # wear time in minutes
air_temp = np.random.normal(298, 2, n_samples)  # Kelvin
process_temp = air_temp + np.random.normal(10, 1, n_samples)
vibration = np.random.exponential(1.5, n_samples).clip(0.1, 8.0)

# Failure conditions logic (failures represent ~5% of events)
# Failure probability increases if torque is high and rpm is low, or if tool wear is extreme, or if vibration is extreme.
p_fail = (
    0.01 
    + 0.25 * (tool_wear > 200) 
    + 0.30 * (vibration > 4.5) 
    + 0.20 * ((torque > 60) & (rpm < 1200))
    + 0.15 * (process_temp > 311)
).clip(0, 1)

failure = np.random.binomial(1, p_fail)

df = pd.DataFrame({
    'RPM': rpm,
    'Torque_Nm': torque,
    'Tool_Wear_Min': tool_wear,
    'Air_Temp_K': air_temp,
    'Process_Temp_K': process_temp,
    'Vibration_mm_s': vibration,
    'Failure': failure
})

print(f"Dataset failures: {df['Failure'].sum()} failures out of {len(df)} samples ({df['Failure'].mean()*100:.1f}%)")
df.head(10)



Dataset failures: 151 failures out of 2000 samples (7.5%)


,RPM,Torque_Nm,Tool_Wear_Min,Air_Temp_K,Process_Temp_K,Vibration_mm_s,Failure
0,1129.557851,29.120134,167.911241,298.424214,308.463909,5.779184,0
1,1283.800202,26.532169,121.215479,295.005983,305.808206,0.976158,0
2,1515.712765,48.829712,58.729297,297.123215,306.820662,0.973648,0
3,1229.172882,30.713056,122.284112,297.626045,306.089223,4.125730,0
4,1625.229692,38.714469,15.805319,293.554526,301.403519,0.829735,0
5,1646.677510,63.297314,178.390633,295.607997,305.000049,1.862114,0
6,1205.804289,30.980871,21.281723,295.991721,305.485238,0.816899,0
7,1164.916090,22.363241,50.927840,297.180865,306.312581,1.548774,0
8,1616.605513,40.055215,202.074335,298.887572,308.374681,0.789015,0
9,1364.507787,29.149510,31.873729,300.072100,310.664060,2.922718,0


## Classification Model Training

We partition our dataset into training and validation folds (80/20 split) and fit an extreme gradient boosted classifier (XGBoost) using standard hyperparameter configurations.



In [2]:
X = df.drop(columns=['Failure'])
y = df['Failure']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Instantiate and fit XGBoost classifier
clf = XGBClassifier(
    n_estimators=100,
    learning_rate=0.08,
    max_depth=5,
    random_state=42,
    eval_metric='logloss'
)
clf.fit(X_train, y_train)

# Predictions evaluation
y_pred = clf.predict(X_test)
y_pred_proba = clf.predict_proba(X_test)[:, 1]

print("Classification Report:")
print(classification_report(y_test, y_pred))



Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.98      0.96       370
           1       0.25      0.07      0.11        30

    accuracy                           0.92       400
   macro avg       0.59      0.53      0.53       400
weighted avg       0.88      0.92      0.89       400



## Feature Importances

We extract relative feature importances from our trained XGBoost classifier to understand which sensor inputs contribute most to machine failure predictions.



In [3]:
feature_imp = pd.Series(clf.feature_importances_, index=X.columns).sort_values(ascending=False)
print("Feature Importances:")
print(feature_imp)



Feature Importances:
Tool_Wear_Min     0.295011
Vibration_mm_s    0.278131
Process_Temp_K    0.154267
Air_Temp_K        0.110044
Torque_Nm         0.087945
RPM               0.074604
dtype: float32


## Interactive Evaluation Plotly Dashboard

We construct a multi-trace interactive Plotly chart showcasing the Model ROC curve (Left) and the Feature Importance profile (Right).



In [4]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Compute ROC curve and AUC area
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
roc_auc = auc(fpr, tpr)

# Convert to standard Python lists to avoid JSON serialization errors
fpr_list = list(fpr)
tpr_list = list(tpr)
feature_imp_vals = [float(v) for v in feature_imp.values[::-1]]
feature_imp_idx = list(feature_imp.index[::-1])

# Initialize a subplots grid: 1 row, 2 columns
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Receiver Operating Characteristic (ROC)', 'Feature Importance Analysis'),
    horizontal_spacing=0.15
)

# 1. ROC Curve Trace (Col 1)
fig.add_trace(
    go.Scatter(
        x=fpr_list, y=tpr_list,
        mode='lines',
        name=f'ROC curve (AUC = {roc_auc:.3f})',
        line=dict(color='#2563EB', width=3)
    ),
    row=1, col=1
)
# Reference baseline line
fig.add_trace(
    go.Scatter(
        x=[0, 1], y=[0, 1],
        mode='lines',
        name='Random Guess',
        line=dict(color='#94A3B8', dash='dash')
    ),
    row=1, col=1
)

# 2. Feature Importances Trace (Col 2)
fig.add_trace(
    go.Bar(
        x=feature_imp_vals,
        y=feature_imp_idx,
        orientation='h',
        name='Feature Importance',
        marker=dict(color='#059669')
    ),
    row=1, col=2
)

# Layout settings
fig.update_layout(
    title_text="XGBoost Predictive Maintenance Model Evaluation",
    template="plotly_white",
    showlegend=True,
    legend=dict(
        x=0.01,
        y=0.01,
        bgcolor='rgba(255, 255, 255, 0.7)'
    ),
    width=900,
    height=480
)

# Axes settings
fig.update_xaxes(title_text="False Positive Rate", row=1, col=1)
fig.update_yaxes(title_text="True Positive Rate", row=1, col=1)
fig.update_xaxes(title_text="Relative Importance Score", row=1, col=2)
fig.update_yaxes(title_text="Sensor Feature", row=1, col=2)

fig.show()
